In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# QESEM: Sebuah Qiskit Function oleh Qedma
*Lihat [referensi API](https://docs.quantum.ibm.com/api/functions/qedma-qesem)*

> **Note:** Qiskit Functions adalah fitur eksperimental yang hanya tersedia untuk pengguna IBM Quantum&reg; Premium Plan, Flex Plan, dan On-Prem (melalui IBM Quantum Platform API) Plan. Fitur ini dalam status rilis pratinjau dan dapat berubah sewaktu-waktu.

## Gambaran Umum
Meskipun unit pemrosesan kuantum telah berkembang pesat dalam beberapa tahun terakhir, error akibat noise dan ketidaksempurnaan di perangkat keras yang ada tetap menjadi tantangan utama bagi pengembang algoritma kuantum. Seiring bidang ini mendekati komputasi kuantum skala utilitas yang tidak dapat diverifikasi secara klasik, solusi untuk menghilangkan noise dengan akurasi terjamin menjadi semakin penting. Untuk mengatasi tantangan ini, Qedma telah mengembangkan Quantum Error Mitigation (QESEM), yang terintegrasi secara mulus di IBM Quantum Platform sebagai [Qiskit Function](/guides/functions).

Dengan QESEM, pengguna bisa menjalankan sirkuit kuantum mereka di QPU yang berisik untuk mendapatkan hasil bebas error yang sangat akurat dengan overhead waktu QPU yang sangat efisien, mendekati batas fundamental. Untuk mencapai ini, QESEM memanfaatkan serangkaian metode proprietary yang dikembangkan oleh Qedma, untuk karakterisasi dan pengurangan error. Teknik pengurangan error mencakup optimisasi gate, transpilasi yang sadar noise, error suppression (ES), dan error mitigation (EM) yang tidak bias. Dengan kombinasi metode berbasis karakterisasi ini, pengguna bisa mendapatkan hasil yang andal dan bebas error untuk sirkuit kuantum bervolume besar yang generik, membuka aplikasi yang sebelumnya tidak bisa dicapai.

Untuk deskripsi lengkap komponen-komponen yang mendasarinya, serta demonstrasi skala utilitas, lihat makalah [Reliable high-accuracy error mitigation for utility-scale quantum circuits](https://arxiv.org/abs/2508.10997).
## Deskripsi
Kamu bisa menggunakan fungsi QESEM oleh Qedma untuk dengan mudah mengestimasi dan mengeksekusi sirkuitmu dengan error suppression dan mitigation, mencapai volume sirkuit yang lebih besar dan akurasi yang lebih tinggi. Untuk menggunakan QESEM, kamu menyediakan sebuah sirkuit kuantum, sekumpulan observabel untuk diukur, target akurasi statistik untuk setiap observabel, dan QPU yang dipilih. Sebelum menjalankan sirkuit ke akurasi target, kamu bisa mengestimasi waktu QPU yang dibutuhkan berdasarkan kalkulasi analitik yang tidak memerlukan eksekusi sirkuit. Setelah puas dengan estimasi waktu QPU, kamu bisa mengeksekusi sirkuit dengan QESEM.

Saat kamu mengeksekusi sebuah sirkuit, QESEM menjalankan protokol karakterisasi perangkat yang disesuaikan dengan sirkuitmu, menghasilkan model noise yang andal untuk error yang terjadi dalam sirkuit. Berdasarkan karakterisasi tersebut, QESEM pertama-tama mengimplementasikan transpilasi yang sadar noise untuk memetakan sirkuit input ke sekumpulan qubit fisik dan gate, yang meminimalkan noise yang memengaruhi observabel target. Ini mencakup gate yang tersedia secara native (CX/CZ di perangkat IBM&reg;), serta gate tambahan yang dioptimalkan oleh QESEM, membentuk extended gate set QESEM. QESEM kemudian menjalankan sekumpulan sirkuit ES dan EM berbasis karakterisasi di QPU dan mengumpulkan hasil pengukurannya. Ini kemudian diproses secara klasik untuk memberikan nilai ekspektasi yang tidak bias dan error bar untuk setiap observabel, sesuai dengan akurasi yang diminta.

![Gambaran umum Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM telah terbukti memberikan hasil akurasi tinggi untuk berbagai aplikasi kuantum dan pada volume sirkuit terbesar yang dapat dicapai saat ini. QESEM menawarkan fitur-fitur berikut yang menghadap pengguna, yang didemonstrasikan di bagian benchmark di bawah:
-	**Akurasi terjamin:** QESEM menghasilkan estimasi yang tidak bias untuk nilai ekspektasi observabel. Metode EM-nya dilengkapi dengan jaminan teoretis, yang — bersama karakterisasi mutakhir Qedma — memastikan mitigasi konvergen ke output sirkuit tanpa noise hingga akurasi yang ditentukan pengguna. Berbeda dengan banyak metode EM heuristik yang rentan terhadap error sistematis atau bias, akurasi terjamin QESEM sangat penting untuk memastikan hasil yang andal pada sirkuit kuantum dan observabel yang generik.
-	**Skalabilitas ke QPU besar:** Waktu QPU QESEM bergantung pada volume sirkuit, tetapi tidak bergantung pada jumlah qubit. Qedma telah mendemonstrasikan QESEM pada perangkat kuantum terbesar yang tersedia saat ini, termasuk IBM Quantum 127-qubit Eagle dan perangkat Heron 133-qubit.
-	**Agnostik aplikasi:** QESEM telah didemonstrasikan pada berbagai aplikasi, termasuk simulasi Hamiltonian, VQE, QAOA, dan estimasi amplitudo. Pengguna bisa memasukkan sirkuit kuantum dan observabel apa pun untuk diukur, dan mendapatkan hasil bebas error yang akurat. Satu-satunya batasan ditentukan oleh spesifikasi perangkat keras dan waktu QPU yang dialokasikan, yang menentukan volume sirkuit yang dapat diakses dan akurasi output. Sebaliknya, banyak solusi pengurangan error yang spesifik aplikasi atau melibatkan heuristik yang tidak terkontrol, sehingga tidak berlaku untuk sirkuit kuantum dan aplikasi yang generik.
-  **Extended gate set:** QESEM mendukung gate dengan sudut fraksional, dan menyediakan gate $Rzz(\theta)$ sudut fraksional yang dioptimalkan Qedma di perangkat IBM Quantum Heron dan Eagle. Extended gate set ini memungkinkan kompilasi yang lebih efisien dan membuka volume sirkuit yang lebih besar hingga faktor 2 dibandingkan kompilasi CX/CZ default.
-	**Observabel multibase:** QESEM mendukung observabel input yang terdiri dari banyak Pauli string yang tidak komuter, seperti Hamiltonian generik. Pemilihan basis pengukuran dan optimisasi alokasi sumber daya QPU (shots dan sirkuit) kemudian dilakukan secara otomatis oleh QESEM untuk meminimalkan waktu QPU yang dibutuhkan untuk akurasi yang diminta. Optimisasi ini, yang memperhitungkan fidelitas perangkat keras dan laju eksekusi, memungkinkan kamu menjalankan sirkuit yang lebih dalam dan mendapatkan akurasi yang lebih tinggi.
## Benchmark
QESEM telah diuji pada berbagai kasus penggunaan dan aplikasi. Contoh-contoh berikut dapat membantu kamu menilai jenis workload apa yang bisa kamu jalankan dengan QESEM.

Ukuran kinerja utama untuk mengkuantifikasi tingkat kesulitan baik error mitigation maupun simulasi klasik untuk sirkuit dan observabel tertentu adalah **volume aktif**: jumlah gate CNOT yang memengaruhi observabel dalam sirkuit. Volume aktif bergantung pada kedalaman dan lebar sirkuit, bobot observabel, dan struktur sirkuit, yang menentukan light cone dari observabel. Untuk detail lebih lanjut, lihat pembicaraan dari [IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM memberikan nilai yang sangat besar di regime volume tinggi, memberikan hasil yang andal untuk sirkuit dan observabel yang generik.

![Volume aktif](../docs/images/guides/qedma-qesem/active_volume.svg)

| Aplikasi | Jumlah qubit | Perangkat | Deskripsi sirkuit | Akurasi | Total waktu | Penggunaan runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Sirkuit VQE  | 8              | Eagle (r3) | 21 total layer, 9 basis pengukuran, rantai 1D                    | 98%      | 35 menit      | 14 menit         |
| Kicked Ising   | 28              | Eagle (r3) | 3 layer unik x 3 langkah, topologi heavy-hex 2D                      | 97%     | 22 menit      | 4 menit          |
| Kicked Ising   | 28              | Eagle (r3) | 3 layer unik x 8 langkah, topologi heavy-hex 2D                     | 97%      | 116 menit      | 23 menit          |
| Simulasi Hamiltonian Trotterisasi   | 40  | Eagle (r3)            | 2 layer unik x 10 langkah Trotter, rantai 1D                    | 97%      | 3 jam     | 25 menit         |
| Simulasi Hamiltonian Trotterisasi   | 119  | Eagle (r3)           | 3 layer unik x 9 langkah Trotter, topologi heavy-hex 2D                    | 95%      | 6,5 jam     | 45 menit         |
| Kicked Ising  | 136             | Heron (r2) | 3 layer unik x 15 langkah, topologi heavy-hex 2D                    | 99%      | 52 menit             | 9 menit           |

Akurasi diukur di sini relatif terhadap nilai ideal observabel: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, di mana '$\epsilon$' adalah presisi absolut mitigasi (ditetapkan oleh input pengguna), dan $\langle O \rangle_{ideal}$ adalah observabel pada sirkuit tanpa noise.
'Penggunaan runtime' mengukur penggunaan benchmark dalam mode batch (jumlah penggunaan job individual), sedangkan 'total waktu' mengukur penggunaan dalam mode sesi (waktu dinding eksperimen), yang mencakup waktu klasik dan komunikasi tambahan. QESEM tersedia untuk eksekusi dalam kedua mode, sehingga pengguna bisa memanfaatkan sumber daya yang tersedia sebaik mungkin.

Sirkuit Kicked Ising 28-qubit mensimulasikan Discrete Time Quasicrystal yang dipelajari oleh Shinjo et al. (lihat [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) dan [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) pada tiga loop terhubung dari ibm_kawasaki. Parameter sirkuit yang digunakan di sini adalah $(\theta_x, \theta_z) = (0.9 \pi, 0)$, dengan keadaan awal feromagnetik $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. Observabel yang diukur adalah nilai absolut magnetisasi $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. Eksperimen Kicked Ising skala utilitas dijalankan pada 136 qubit terbaik dari ibm_fez; benchmark khusus ini dijalankan pada sudut Clifford $(\theta_x, \theta_z) = (\pi, 0)$, di mana volume aktif tumbuh perlahan seiring kedalaman sirkuit, yang — bersama fidelitas perangkat yang tinggi — memungkinkan akurasi tinggi pada waktu runtime yang singkat.

Sirkuit simulasi Hamiltonian Trotterisasi untuk model Ising Transverse-Field pada sudut fraksional: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ dan $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ secara berurutan (lihat [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Sirkuit skala utilitas dijalankan pada 119 qubit terbaik dari ibm_brisbane, sedangkan eksperimen 40-qubit dijalankan pada rantai terbaik yang tersedia. Akurasi dilaporkan untuk magnetisasi; hasil akurasi tinggi juga diperoleh untuk observabel dengan bobot yang lebih tinggi.

Sirkuit VQE dikembangkan bersama peneliti dari Center for Quantum Technology and Applications di Deutsches Elektronen-Synchrotron (DESY). Observabel target di sini adalah Hamiltonian yang terdiri dari sejumlah besar Pauli string yang tidak komuter, menekankan kinerja QESEM yang dioptimalkan untuk observabel multi-basis. Mitigasi diterapkan pada ansatz yang dioptimalkan secara klasik; meskipun hasil ini masih belum dipublikasikan, hasil dengan kualitas yang sama akan diperoleh untuk sirkuit yang berbeda dengan properti struktural yang serupa.
## Memulai
Autentikasi menggunakan [kunci API IBM Quantum Platform](http://quantum.cloud.ibm.com/)-mu, dan pilih Qiskit Function QESEM sebagai berikut. (Cuplikan ini mengasumsikan kamu sudah [menyimpan akunmu](/guides/functions#install-qiskit-functions-catalog-client) ke lingkungan lokalmu.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog


catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# load the function
qesem_function = catalog.load("qedma/qesem")

## Examples

To get started, try this basic example of estimating the required QPU time to run QESEM for a given `pub`:

In [3]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [4]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Kamu bisa menggunakan API Qiskit Serverless yang familiar untuk memeriksa status workload Qiskit Function-mu atau mengembalikan hasilnya:

In [5]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Cuplikan kode berikut menjelaskan cara mengambil estimasi waktu QPU (saat `estimate_time_only` diset):

In [6]:
# Print the ID so you can use it later, if necessary
print(sample_job.job_id)

print(sample_job.status())
result = sample_job.result()

9a87a23f-82f5-429e-91fb-cc8a9d107980
QUEUED


Cuplikan kode berikut mendemonstrasikan cara mengambil hasil mitigasi (saat `estimate_time_only` tidak diset) dan metrik eksekusi. Ini berisi data penting yang memungkinkan pemahaman lebih mendalam tentang bagaimana parameter berbeda memengaruhi eksekusi QESEM. Ini juga mungkin relevan saat menulis makalah berdasarkan penelitianmu.

In [7]:
print(
    f"The estimated QPU time for this PUB is: "
    f"\n{time_estimate_result[0].metadata}"
)

The estimated QPU time for this PUB is: 
{'time_estimation_sec': 1800, 'description': None, 'instance': 'crn:v1:bluemix:public:quantum-computing:us-east:a/6c63dae5281147f1a0449b36e0aaba3a:ae40ab55-8c55-4042-9204-71a6541d56ec::', 'transpilation_level': 'standard', 'parallel_execution': False, 'total_qpu_time': 0, 'empirical_estimation_mitigation_results': None, 'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 42.44837867887691, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 17.879877626895905, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: EXECUTING_QPU': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 0.0, 'GPU_TIME': 0.0, 'QPU_TIME': 0.0}}}


## Ambil pesan error
Jika status workloadmu ERROR, gunakan `job.result()` untuk mengambil pesan error sebagai berikut:

In [9]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: "
    f"\n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n "
    f"{results.metadata['total_shots']} / "
    f"{results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    if "qubit_map" in circuit:
        print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

Mitigated expectation values: 
[1.00169764 1.00460812]
Mitigated error-bar: 
[0.00155021 0.0099558 ]
Noisy expectation values: 
[0.95717143 0.94271429]
Noisy error-bar: 
[0.00206982 0.00872689]
Total QPU time: 
 150.0
Gates fidelity measured during the experiment: 
 {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}
Total shots / mitigation shots: 
 495600 / 220400
Transpiled circuits:
Circuit 0:
  Circuit: 
 OPENQASM 3.0;
include "stdgates.inc";
bit[145] c0;
qubit[145] q0;
rz(-pi) q0[143];
rz(0) q0[141];
rz(-pi) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(-pi/2) q0[143];
rz(pi/2) q0[141];
rz(-pi/2) q0[140];
sx q0[143];
sx q0[141];
sx q0[140];
rz(pi/2) q0[143];
rz(-pi/2) q0[141];
rz(pi/2) q0[140];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
cz q0[144], q0[143];
cz q0[142], q0[141];
barrier q0[144], q0[143], q0[142], q0[141];
barrier q0[140], q0[141], q0[142], q0[143], q0[144];
rz(-pi) q0[144];
rz(-pi/2) q0[143];
rz(0) q0[142];
rz(-pi/2) q0[141];
sx q0[144];
sx q0[143];

## Fetch error messages

If your workload status is ERROR, use `job.result()` to fetch the error message to fetch the error message as follows:

In [14]:
# Get the result and truncate for readability
result = sample_job.result()
result_str = str(result)
max_length = 500  # Adjust this value as necessary

if len(result_str) > max_length:
    truncated = (
        result_str[:max_length]
        + f"... (truncated {len(result_str) - max_length} characters)"
    )
else:
    truncated = result_str

print(truncated)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(2,), dtype=float64>), stds=np.ndarray(<shape=(2,), dtype=float64>), shape=(2,)), metadata={'gate_fidelities': {'CZ': 0.9979051606662408, 'ID1Q': 0.9993865847216725}, 'total_shots': 495600, 'mitigation_shots': 220400, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[145] c0;\nqubit[145] q0;\nrz(-pi) q0[143];\nrz(0) q0[141];\nrz(-pi) q0[140];\nsx q0[143];\nsx q0[141];\nsx q0[140];\nrz(-pi/2) q0[143];\nrz(pi... (truncated 4174 characters)


## Dapatkan dukungan

Tim dukungan Qedma siap membantu! Jika kamu mengalami masalah atau memiliki pertanyaan tentang penggunaan Qiskit Function QESEM, jangan ragu untuk menghubungi kami. Staf dukungan kami yang berpengetahuan dan ramah siap membantu kamu dengan masalah teknis atau pertanyaan apa pun yang kamu miliki.

Kamu bisa menghubungi kami melalui email di support@qedma.com untuk bantuan. Sertakan detail sebanyak mungkin tentang masalah yang kamu alami untuk membantu kami memberikan respons yang cepat dan akurat. Kamu juga bisa menghubungi perwakilan POC Qedma yang ditugaskan untukmu melalui email atau telepon.

Untuk membantu kami membantu kamu lebih efisien, silakan berikan informasi berikut saat menghubungi kami:

- Deskripsi detail tentang masalah tersebut
- ID job
- Pesan error atau kode apa pun yang relevan

Kami berkomitmen untuk memberikan dukungan yang cepat dan efektif untuk memastikan kamu mendapatkan pengalaman terbaik dengan Qiskit Function kami.

Kami selalu berusaha meningkatkan produk kami dan kami menyambut saranmu! Jika kamu memiliki ide tentang bagaimana kami bisa meningkatkan layanan atau fitur yang ingin kamu lihat, silakan kirimkan pemikiranmu ke support@qedma.com.

## Langkah selanjutnya

> **Tip:** - [Minta akses ke Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem).
> - Kunjungi [referensi API](https://docs.quantum.ibm.com/api/functions/qedma-qesem) untuk Qiskit Function ini.
> - Tinjau [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199](https://arxiv.org/pdf/2507.01199).
> - Tinjau [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997](https://arxiv.org/pdf/2508.10997).